In [10]:
# CELL 1: Imports & basic setup

import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [11]:
# CELL 2: Mount Google Drive & define paths

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/mbti-tune")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

MBTI_PLAYLIST_DIR = RAW_DIR / "mbti_playlists"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MBTI_PLAYLIST_DIR:", MBTI_PLAYLIST_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/mbti-tune
MBTI_PLAYLIST_DIR: /content/drive/MyDrive/mbti-tune/data/raw/mbti_playlists


In [12]:
# CELL 3: Load aggregated MBTI playlist stats (16 CSV files)

csv_files = sorted(glob.glob(str(MBTI_PLAYLIST_DIR / "*.csv")))
print("Found aggregated MBTI files:", len(csv_files))

agg_rows = []
for f in csv_files:
    df = pd.read_csv(f)
    agg_rows.append(df)

agg_df = pd.concat(agg_rows, ignore_index=True)
print("Aggregated MBTI stats shape:", agg_df.shape)
print(agg_df.head())

assert "mbti" in agg_df.columns, "Expected 'mbti' column in aggregated MBTI files"


Found aggregated MBTI files: 16
Aggregated MBTI stats shape: (4201, 49)
   mbti function_pair                                 playlist_name  \
0  ENFJ            NF                                        Enfj 😇   
1  ENFJ            NF                      enfj // the protagonist    
2  ENFJ            NF  enfj songs according to personality database   
3  ENFJ            NF                                  ENFJ vibes‼️   
4  ENFJ            NF                                        [ENFJ]   

              playlist_id  track_count  danceability_mean  danceability_stdev  \
0  4AE4DBt4YjJJ8v4Hk9myWl           41           0.580512            0.124325   
1  6th4JW4Dky7UjeEhHuQeQd           38           0.550947            0.156753   
2  4g2AYw35pR37u2QpwpSryq           50           0.564680            0.116814   
3  1eVgLeDoHD123LB6VldjGY           50           0.535200            0.122586   
4  49BWXniDb2gtfI1e4Z3ILr           50           0.526700            0.123802   

   energy_mean

In [13]:
# CELL 4: Data Cleaning — handle NaN, inf, scaling

# Select numeric columns only
numeric_cols = agg_df.select_dtypes(include=[np.number]).columns
print("Numeric feature columns:", len(numeric_cols))

X_raw = agg_df[numeric_cols].values.astype(np.float32)
y_raw = agg_df["mbti"].astype(str)

# 1. Replace NaN, inf, -inf
X_clean = np.nan_to_num(X_raw, nan=0.0, posinf=0.0, neginf=0.0)

# 2. Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X_clean).astype(np.float32)

# Encode MBTI labels
unique_types = sorted(y_raw.unique())
type_to_idx = {t: i for i, t in enumerate(unique_types)}
idx_to_type = {i: t for t, i in type_to_idx.items()}

y = y_raw.map(type_to_idx).values.astype(np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique MBTI types:", unique_types)


Numeric feature columns: 45
X shape: (4201, 45)
y shape: (4201,)
Unique MBTI types: ['ENFJ', 'ENFP', 'ENTJ', 'ENTP', 'ESFJ', 'ESFP', 'ESTJ', 'ESTP', 'INFJ', 'INFP', 'INTJ', 'INTP', 'ISFJ', 'ISFP', 'ISTJ', 'ISTP']


In [14]:
# CELL 5: Dataset & DataLoader

class MBTIDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = MBTIDataset(X, y)

batch_size = 8
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

print("Batches:", len(loader))


Batches: 526


In [15]:
# CELL 6: Define neural MBTI classifier (no BatchNorm)

class MBTIClassifierV3(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_classes=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

input_dim = X.shape[1]
num_classes = len(unique_types)

model = MBTIClassifierV3(input_dim=input_dim, hidden_dim=128, num_classes=num_classes).to(device)
print(model)


MBTIClassifierV3(
  (net): Sequential(
    (0): Linear(in_features=45, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=16, bias=True)
  )
)


In [16]:
# CELL 7: Training setup

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

num_epochs = 200

print("num_epochs:", num_epochs)
print("input_dim:", input_dim)
print("num_classes:", num_classes)


num_epochs: 200
input_dim: 45
num_classes: 16


In [17]:
# CELL 8: Train classifier on all MBTI rows (no validation)

best_loss = float("inf")
best_state = None

for epoch in range(1, num_epochs + 1):
    model.train()
    train_losses = []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    train_loss = np.mean(train_losses)

    if train_loss < best_loss:
        best_loss = train_loss
        best_state = model.state_dict()

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | train_loss={train_loss:.4f}")

model.load_state_dict(best_state)
print("\n✅ Training finished. Best training loss:", best_loss)


Epoch 001 | train_loss=2.7253
Epoch 020 | train_loss=2.1880
Epoch 040 | train_loss=2.0713
Epoch 060 | train_loss=1.9818
Epoch 080 | train_loss=1.9128
Epoch 100 | train_loss=1.8380
Epoch 120 | train_loss=1.7705
Epoch 140 | train_loss=1.7125
Epoch 160 | train_loss=1.6648
Epoch 180 | train_loss=1.5931
Epoch 200 | train_loss=1.5550

✅ Training finished. Best training loss: 1.5482419087859614


In [18]:
# CELL 9: Evaluate on training set

model.eval()
with torch.no_grad():
    X_t = torch.from_numpy(X).to(device)
    logits = model(X_t)
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = np.argmax(probs, axis=1)

train_acc = accuracy_score(y, preds)
print("Training accuracy:", train_acc)

# MAE between true one-hot and predicted probabilities
y_onehot = np.zeros_like(probs)
for i, label in enumerate(y):
    y_onehot[i, label] = 1.0

global_mae = mean_absolute_error(y_onehot, probs)
print("Global MAE (one-hot vs predicted probs):", global_mae)

print("\nClassification report:")
print(classification_report(y, preds, target_names=[idx_to_type[i] for i in range(num_classes)]))


Training accuracy: 0.6386574625089264
Global MAE (one-hot vs predicted probs): 0.07651364058256149

Classification report:
              precision    recall  f1-score   support

        ENFJ       0.63      0.65      0.64       302
        ENFP       0.61      0.63      0.62       302
        ENTJ       0.64      0.80      0.71       303
        ENTP       0.75      0.65      0.70       303
        ESFJ       0.64      0.49      0.55       115
        ESFP       0.61      0.53      0.57       231
        ESTJ       0.76      0.46      0.57       122
        ESTP       0.68      0.75      0.71       306
        INFJ       0.58      0.70      0.63       304
        INFP       0.62      0.60      0.61       295
        INTJ       0.61      0.64      0.63       307
        INTP       0.69      0.69      0.69       299
        ISFJ       0.54      0.67      0.60       221
        ISFP       0.67      0.62      0.64       295
        ISTJ       0.67      0.42      0.52       196
        ISTP

In [19]:
# CELL 10: Dominant MBTI letters with percentages

indices_E = [i for i, t in idx_to_type.items() if t[0] == "E"]
indices_I = [i for i, t in idx_to_type.items() if t[0] == "I"]
indices_S = [i for i, t in idx_to_type.items() if t[1] == "S"]
indices_N = [i for i, t in idx_to_type.items() if t[1] == "N"]
indices_T = [i for i, t in idx_to_type.items() if t[2] == "T"]
indices_F = [i for i, t in idx_to_type.items() if t[2] == "F"]
indices_J = [i for i, t in idx_to_type.items() if t[3] == "J"]
indices_P = [i for i, t in idx_to_type.items() if t[3] == "P"]

def letter_probs(prob_vec):
    return {
        "E": float(prob_vec[indices_E].sum()),
        "I": float(prob_vec[indices_I].sum()),
        "S": float(prob_vec[indices_S].sum()),
        "N": float(prob_vec[indices_N].sum()),
        "T": float(prob_vec[indices_T].sum()),
        "F": float(prob_vec[indices_F].sum()),
        "J": float(prob_vec[indices_J].sum()),
        "P": float(prob_vec[indices_P].sum()),
    }

print("Dominant-letter MBTI predictions:\n")

for i in range(len(X)):
    true_type = idx_to_type[y[i]]
    prob_vec = probs[i]
    pred_idx = np.argmax(prob_vec)
    pred_type = idx_to_type[pred_idx]

    lp = letter_probs(prob_vec)

    EI = "E" if lp["E"] >= lp["I"] else "I"
    SN = "S" if lp["S"] >= lp["N"] else "N"
    TF = "T" if lp["T"] >= lp["F"] else "F"
    JP = "J" if lp["J"] >= lp["P"] else "P"

    EI_pct = round(100 * max(lp["E"], lp["I"]), 1)
    SN_pct = round(100 * max(lp["S"], lp["N"]), 1)
    TF_pct = round(100 * max(lp["T"], lp["F"]), 1)
    JP_pct = round(100 * max(lp["J"], lp["P"]), 1)

    final_mbti = EI + SN + TF + JP

    print(f"Sample {i}:")
    print(f"  True: {true_type} | Pred: {pred_type}")
    print(f"  Dominant-letter MBTI: {final_mbti}")
    print(f"    {EI} {EI_pct}%")
    print(f"    {SN} {SN_pct}%")
    print(f"    {TF} {TF_pct}%")
    print(f"    {JP} {JP_pct}%\n")


Streaming output truncated to the last 5000 lines.
Sample 3576:
  True: ISFP | Pred: ISFP
  Dominant-letter MBTI: ISFP
    I 82.5%
    S 78.2%
    F 81.1%
    P 77.0%

Sample 3577:
  True: ISFP | Pred: ENFP
  Dominant-letter MBTI: ENFP
    E 89.7%
    N 50.4%
    F 94.6%
    P 93.0%

Sample 3578:
  True: ISFP | Pred: ISFP
  Dominant-letter MBTI: ISFP
    I 94.0%
    S 71.4%
    F 70.6%
    P 71.0%

Sample 3579:
  True: ISFP | Pred: ISFP
  Dominant-letter MBTI: ISFP
    I 78.4%
    S 84.5%
    F 54.1%
    P 92.9%

Sample 3580:
  True: ISFP | Pred: ISFP
  Dominant-letter MBTI: ESFP
    E 55.0%
    S 69.8%
    F 55.3%
    P 77.9%

Sample 3581:
  True: ISFP | Pred: ISFP
  Dominant-letter MBTI: ISFP
    I 92.5%
    S 93.5%
    F 95.8%
    P 94.6%

Sample 3582:
  True: ISFP | Pred: INFP
  Dominant-letter MBTI: ISFP
    I 73.8%
    S 52.2%
    F 94.6%
    P 75.1%

Sample 3583:
  True: ISFP | Pred: ISFP
  Dominant-letter MBTI: ISFP
    I 83.9%
    S 84.2%
    F 81.8%
    P 87.8%

Sample 3584:


In [20]:
# CELL 11: Final classifier summary

print("===== FINAL MBTI CLASSIFIER V3 SUMMARY =====")
print("Training samples:", X.shape[0])
print("Input dim (aggregated features):", input_dim)
print("Num MBTI classes:", num_classes)
print("Training accuracy:", train_acc)
print("Global MAE:", global_mae)
print("Aggregated MBTI dir:", MBTI_PLAYLIST_DIR)
print("===========================================")


===== FINAL MBTI CLASSIFIER V3 SUMMARY =====
Training samples: 4201
Input dim (aggregated features): 45
Num MBTI classes: 16
Training accuracy: 0.6386574625089264
Global MAE: 0.07651364058256149
Aggregated MBTI dir: /content/drive/MyDrive/mbti-tune/data/raw/mbti_playlists
